# Exercicis — El dataset digits: xifres escrites a mà

**Optativa d'Aprenentatge automàtic · DAM/DAW 2n**

Fins ara les columnes dels vostres datasets eren mesures amb nom: llargada
d'un pètal, grau d'alcohol, sucre residual. Cadascuna volia dir alguna cosa
per si sola.

`digits`, de scikit-learn, és diferent: **1797 imatges de 8×8 píxels** amb
xifres escrites a mà (0 al 9), i **cada columna és un píxel**. Cap columna
té sentit per si sola —«la columna 27» no vol dir res—, i tot i així **els
mateixos cinc models que ja coneixeu funcionen igual de bé**. Aquest és el
salt d'aquest quadern: una imatge també és una taula.

Els exercicis fan servir `sklearn.datasets.load_digits`. Recordeu escalar
les dades amb els models que ho necessiten (com heu vist als quaderns de
k-NN i SVM) i fixar `random_state=42` a tot arreu on calgui.

## Les eines que necessites

Aquest quadern demana aquestes eines. Aquí les tens llistades, no
encadenades: quina va abans de quina i què li passes a cada una és la feina
dels exercicis.

- De `sklearn.datasets`: `load_digits`.
- De `sklearn.model_selection`: `train_test_split`.
- De `sklearn.preprocessing` i `sklearn.pipeline`: `StandardScaler`,
  `make_pipeline`.
- Els cinc models de sempre: `KNeighborsClassifier`,
  `DecisionTreeClassifier`, `RandomForestClassifier`, `LogisticRegression`,
  `SVC`.
- De `sklearn.metrics`: `confusion_matrix` i `ConfusionMatrixDisplay`.
- De `sklearn.feature_selection`: `SelectKBest` i `f_classif`.
- De pandas: `pd.DataFrame`.
- De NumPy, per moure't entre imatge i taula i per remenar la matriu de
  confusió: `.reshape`, `np.array_equal`, `np.fill_diagonal`,
  `np.unravel_index`, `np.argsort`, `np.random.default_rng`.

La matriu de confusió no ha sortit a cap quadern anterior. Cada una
d'aquestes eines està explicada a `EX_00_caixa_eines.ipynb`, amb la sintaxi,
què li has de donar, què et torna i el parany que té.

## Els objectes que manegues, i què els pots demanar

Les eines de la llista de dalt no treballen soltes: cadascuna et torna un
**objecte**, i el pas següent de l'exercici surt de demanar-li alguna cosa a aquell
objecte. Aquí tens, de cada un, el que faràs servir de veritat en aquest quadern:
els **atributs** (sense parèntesis) i els **mètodes** (amb parèntesis).

**El `Bunch` que torna `load_digits()`** — `sklearn.utils.Bunch`, una mena de
diccionari al qual també pots accedir amb punt (`digits.data` és el mateix que
`digits["data"]`):
- `.data` — la taula de 1797×64: una fila per imatge, una columna per píxel
- `.images` — les **mateixes** dades amb una altra forma, 1797×8×8. És l'atribut
  que et deixa dibuixar la xifra sense fer cap `reshape`
- `.target` — les 1797 etiquetes (0 a 9)
- `.target_names` — els 10 noms de classe
- `.feature_names` — els 64 noms de píxel (`pixel_0_0`, `pixel_0_1`...)
- `.DESCR` — el text que descriu el dataset (`print(digits.DESCR)` per llegir-lo)

**L'`ndarray` de NumPy** — `digits.data`, `digits.images[0]`, `X_entrena`,
`prediccions`, la matriu de confusió, `feature_importances_`:
- `.shape` — la tupla amb la forma. `digits.data.shape` és `(1797, 64)` i
  `digits.images.shape` és `(1797, 8, 8)`: la mateixa informació, dues formes
- `.dtype` — el tipus dels valors que hi ha a dins
- `.T` — la matriu transposada
- `.reshape(8, 8)` i `.reshape(-1)` — les dues direccions entre imatge i fila de
  taula. El `-1` vol dir «calcula tu aquesta dimensió»
- `.mean(axis=0)`, `.std(axis=0)` — per columna; amb `axis=1`, per fila
- `.argmax()` — la **posició** del valor més gran dins de l'array aplanat, no el
  valor. Per convertir aquella posició en (fila, columna) hi ha
  `np.unravel_index(...)`
- `.sum()`, `.copy()` — la suma de tot, i una còpia que pots fer malbé sense tocar
  l'original (és el que necessites abans de `np.fill_diagonal`)

**El `DataFrame` de pandas** — `taula`:
- `.shape`, `.columns`, `.dtypes` (atributs)
- `.sort_values("...", ascending=False)`, `.head()` (mètodes)
- `taula.iloc[0]` és la primera fila; `taula.iloc[0]["model"]` en treu una casella

**Un model de scikit-learn** — `KNeighborsClassifier`, `DecisionTreeClassifier`,
`RandomForestClassifier`, `LogisticRegression`, `SVC`. Tots tenen **la mateixa
interfície**, i això és mitja assignatura:
- `.fit(X, y)` — entrena amb aquestes dades; torna el model mateix
- `.predict(X)` — un array amb una etiqueta predita per fila
- `.predict_proba(X)` — una matriu de `(files, 10)` amb la probabilitat de cada
  classe; la columna `i` correspon a `.classes_[i]`
- `.score(X, y)` — la precisió sobre aquestes dades, en un sol número
- `.classes_` — les 10 classes que ha vist en entrenar, ordenades
- `.n_features_in_` — quantes columnes esperava (64, o 20 després del
  `SelectKBest`); si li dones un nombre diferent a `.predict()`, peta
- `.feature_importances_` — **només** els models d'arbre: 64 pesos, un per píxel,
  que sumen 1. Fer-hi `.reshape(8, 8)` els torna a posar en forma d'imatge
- `.coef_` — **només** `LogisticRegression` (i un `SVC` lineal)

**El `Pipeline` que torna `make_pipeline(...)`** — per fora es comporta com un
model qualsevol (`.fit()`, `.predict()`, `.score()`), i per dins guarda els passos:
- `.named_steps` — un diccionari amb els passos pel seu nom
  (`pipe.named_steps["svc"]`, `pipe.named_steps["standardscaler"]`), que és com
  arribes als atributs apresos de cada un
- `.steps` — la llista de parelles (nom, objecte)
- quan crides `.fit()` del pipeline, cada pas s'ajusta amb la sortida de l'anterior:
  és el que et garanteix que l'escalador no ha vist mai l'examen

**El `SelectKBest`** — no és un model, és un **transformador**:
- `.fit(X, y)`, `.transform(X)`, i els dos de cop amb `.fit_transform(X, y)`. No té
  `.predict()` ni `.score()`
- `.get_support()` — un array de 64 booleans: `True` a les columnes que ha triat.
  `.sum()` te'n diu quantes són, i `.reshape(8, 8)` te les ensenya col·locades a la
  imatge
- `.scores_` — la puntuació que ha donat a **cada** una de les 64 columnes, també
  les que ha descartat
- `.get_feature_names_out()` — els noms de les columnes que sobreviuen

**L'`StandardScaler`** — el mateix patró de transformador, amb `.mean_` i
`.scale_`: la mitjana i la desviació de cada columna que ha après de l'entrenament.
Compte amb un detall d'aquest dataset: hi ha píxels que són sempre 0, i per a
aquests l'escalador deixa `.scale_` a 1 en lloc de dividir per zero.

**L'`Axes` de matplotlib** — el que et torna `plt.subplots(4, 5)` dins de l'array
`axs`:
- `.imshow(imatge, cmap="gray")` — dibuixa una matriu 2D com una imatge
- `.set_title(text)`, `.set_xlabel(text)`, `.axis("off")`
- `axs.ravel()` t'aplana la graella d'eixos per recórrer-la amb un sol bucle

Dues regles que valen per a tot el quadern, perquè són les que fan petar el codi
sense que sembli que hi hagi res malament:

- **Un atribut va sense parèntesis i un mètode amb parèntesis.** `model.coef_` et
  dona els coeficients; `model.coef_()` dona `TypeError`, perquè no és una funció.
  I a l'inrevés: el `.describe` d'un DataFrame sense parèntesis no calcula res,
  t'ensenya el mètode.
- **El guió baix al final vol dir «això ho he après de les dades».** `.coef_`,
  `.feature_importances_` i `.classes_` **no existeixen abans de cridar `.fit()`**:
  si els demanes abans, salta un error. El mateix passa amb els atributs apresos
  de qualsevol transformador.

Per interrogar qualsevol d'aquests objectes pel teu compte —`type()`, `dir()`,
`objecte.metode?` i l'autocompletat amb TAB— tens
`FO_00_objectes_i_autocompletar.ipynb`.

## Exercici 1 — Mirar les dades

Carrega el dataset amb `load_digits()`. Mira quina forma té `digits.data`
(files i columnes) i quantes classes diferents hi ha a `digits.target`.
Després dibuixa una graella d'unes 20 xifres (per exemple 4 files × 5
columnes) amb `plt.imshow(digits.images[i], cmap="gray")`, posant
`digits.target[i]` com a títol de cada requadre.

**Com saps que ho has fet bé**: `digits.data` ha de tenir forma
`(1797, 64)`, hi ha d'haver 10 classes (0 a 9), i les xifres dibuixades han
de coincidir a ull amb el número del seu títol.

In [ ]:
# Exercici 1
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt

digits = load_digits()

# imprimeix la forma de digits.data (.shape) i el nombre de classes de
# digits.target (len(set(digits.target)), o digits.target_names)


# dibuixa una graella d'unes 20 xifres amb plt.imshow(digits.images[i], cmap="gray")
# posa digits.target[i] com a titol de cada requadre (plt.subplots amb 4 files x 5 columnes)

## Exercici 2 — De la imatge a la taula

Tria una xifra (per exemple la de l'índex `0`). Imprimeix la seva matriu
8×8 de números (`digits.images[0]`, valors de 0 a 16: la intensitat de
gris) i, al costat, dibuixa-la amb `imshow`. Que es vegin totes dues coses
alhora: la graella de números i el dibuix que en surt.

Després «aplana» aquesta imatge (`.reshape(-1)`) i comprova amb
`np.array_equal` que el resultat és **exactament** `digits.data[0]`.
Aplanar una imatge vol dir posar les 8 files una darrere l'altra fins a
tenir 64 columnes: no és una aproximació, és la mateixa informació escrita
d'una altra manera.

**Com saps que ho has fet bé**: `np.array_equal(...)` ha de donar `True`.

In [ ]:
# Exercici 2
import numpy as np

idx = 0

# imprimeix digits.images[idx] (la matriu 8x8 de numeros)


# dibuixa digits.images[idx] amb imshow (cmap="gray")


# aplana digits.images[idx] amb .reshape(-1) i compara amb digits.data[idx]
# fent servir np.array_equal

## Exercici 3 — El flux de sempre

Separa `digits.data` i `digits.target` en 70 % entrenament i 30 % examen
amb `train_test_split(..., test_size=0.3, random_state=42,
stratify=digits.target)`.

Entrena els cinc models que ja coneixeu —k-NN, arbre de decisió, bosc
aleatori, regressió logística i SVM—, escalant amb `StandardScaler` dins
d'un `Pipeline` els que ho necessiten (k-NN, regressió logística i SVM
treballen amb distàncies o amb coeficients que depenen de l'escala; arbre
i bosc no). Munta una taula (`pd.DataFrame`) amb la precisió de cadascun a
l'examen, ordenada de més a menys.

**Com saps que ho has fet bé**: la taula ha de tenir cinc files, una per
model, ordenada de més a menys precisió. Afegeix-hi una segona columna amb
la precisió sobre `X_entrena`: si a un model li surt 100 % a l'entrenament i
força menys a l'examen, està memoritzant, i això explica la seva posició
abans que el compares amb els altres. Si un model que treballa amb
distàncies o amb coeficients et queda molt per sota de la resta, mira si té
l'`StandardScaler` dins del `Pipeline`.

In [ ]:
# Exercici 3
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import pandas as pd

X = digits.data
y = digits.target

# separa en entrenament/examen amb train_test_split(X, y, test_size=0.3,
# random_state=42, stratify=y)


models = {
    "kNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Arbre": DecisionTreeClassifier(random_state=42),
    "Bosc": RandomForestClassifier(random_state=42),
    "Regressió logística": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=5000, random_state=42)
    ),
    "SVM": make_pipeline(StandardScaler(), SVC(random_state=42)),
}

# recorre models.items(): entrena cada model amb .fit(X_entrena, y_entrena),
# guarda l'objecte ja entrenat a un diccionari `models_entrenats` (el
# necessites a l'exercici 4) i mesura la precisio a l'examen amb
# .score(X_examen, y_examen)

# munta-ho en una taula (pd.DataFrame) ordenada de mes a menys precisio amb
# .sort_values("precisio", ascending=False)

## Exercici 4 — La matriu de confusió de 10×10

Agafa el millor model de l'exercici 3 i calcula la seva matriu de
confusió a l'examen (`confusion_matrix` o `ConfusionMatrixDisplay`).
Amb 10 classes en lloc de 2 o 3, la matriu és de 10×10 en comptes de
2×2 o 3×3. Dibuixa-la i, mirant **fora de la diagonal**, troba la parella
de xifres que es confon més sovint. Escriu quina és en una cel·la de text.

**Com saps que ho has fet bé**: has d'acabar amb dues xifres concretes
identificades per escrit, no només amb el dibuix.

In [ ]:
# Exercici 4
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# agafa el nom del millor model de la `taula` de l'exercici 3
# (si l'has ordenada de mes a menys precisio, es taula.iloc[0]["model"])
# i recupera l'objecte corresponent de `models_entrenats`


# prediccions del millor model sobre X_examen (.predict(X_examen))


# calcula la matriu de confusio (confusion_matrix(y_examen, prediccions))
# dibuixa-la amb ConfusionMatrixDisplay(...).plot()


# fora de la diagonal, troba la posicio (real, predit) amb mes confusions
# pista: np.fill_diagonal(matriu.copy(), 0) i despres np.unravel_index(matriu.argmax(), matriu.shape)

## Exercici 5 — Mirar els errors amb els ulls

Troba els índexs de l'examen on el millor model s'ha equivocat (compara
`prediccions` amb `y_examen`). Reconstrueix la imatge de cada error amb
`X_examen[i].reshape(8, 8)` —el mateix truc de l'exercici 2, a l'inrevés—
i dibuixa'n una graella (per exemple 10) amb el títol `"real: X · predit:
Y"`.

**Com saps que ho has fet bé**: `len(errors_idx)` ha de quadrar amb la
precisió de l'exercici 3: `(len(y_examen) - len(errors_idx)) / len(y_examen)`
ha de donar el mateix número. A cada requadre de la graella, el «real» i el
«predit» han de ser diferents; si et surt un amb els dos iguals, no has
filtrat bé els errors.

In [ ]:
# Exercici 5

# troba els indexs de X_examen on prediccions != y_examen
# (np.where(...)[0] et dona les posicions) i desa'ls a `errors_idx`


# imprimeix quants errors hi ha sobre el total: len(errors_idx) de len(y_examen)


# dibuixa una graella amb (fins a) 10 dels errors:
# imatge = X_examen[i].reshape(8, 8)
# titol = f"real: {y_examen[i]} · predit: {prediccions[i]}"

## Exercici 6 — Quants píxels calen de veritat

Fes servir `SelectKBest(f_classif, k=20)` per quedar-te només amb les 20
columnes (píxels) més informatives, ajustat amb `X_entrena`/`y_entrena`.
Entrena el mateix tipus de model (per exemple SVM, dins d'un `Pipeline`
amb `StandardScaler`) amb les 64 columnes originals i, per separat, amb
només aquestes 20, i compara precisió **i temps d'entrenament** (amb
`time.time()`). Finalment, dibuixa `selector.get_support().reshape(8, 8)`
com una imatge per veure quins píxels ha triat.

**Com saps que ho has fet bé**: la matriu d'entrenament reduïda ha de tenir
20 columnes i `selector.get_support().sum()` ha de donar 20. Cronometra
només la crida a `.fit()`, no la càrrega de dades, i mesura-la dues vegades:
si els dos temps et surten molt diferents, fes la mitjana de tres abans de
comparar res. Per saber si la tria del `SelectKBest` aporta alguna cosa,
repeteix-ho amb 20 columnes agafades a l'atzar
(`np.random.default_rng(42).choice(64, 20, replace=False)`) i compara els
tres resultats.

In [ ]:
# Exercici 6
from sklearn.feature_selection import SelectKBest, f_classif
import time

# entrena i cronometra un make_pipeline(StandardScaler(), SVC()) amb les 64
# columnes: time.time() abans i despres del .fit(X_entrena, y_entrena), i
# la precisio amb .score(X_examen, y_examen)


# selecciona les 20 columnes mes informatives amb SelectKBest(f_classif, k=20),
# desa l'objecte a `selector` (el necessites al final de la cel.la):
# selector.fit_transform(X_entrena, y_entrena) i despres
# selector.transform(X_examen) (l'examen NO es torna a ajustar)


# entrena i cronometra el mateix tipus de model amb nomes aquestes 20
# columnes (un make_pipeline nou, cronometrat igual que abans)


# compara precisio i temps de les dues versions


# comprova selector.get_support().sum() i dibuixa
# selector.get_support().reshape(8, 8) com una imatge (plt.imshow)

## Exercici 7 (opcional) — La forma que mira un bosc aleatori

Entrena un `RandomForestClassifier` amb `X_entrena`/`y_entrena` i dibuixa
`rf.feature_importances_.reshape(8, 8)` com una imatge (per exemple amb
`plt.imshow(..., cmap="hot")`). Cada valor diu com d'important és aquell
píxel per al bosc a l'hora de decidir.

**Com saps que ho has fet bé**: `rf.feature_importances_` ha de tenir 64
valors i sumar 1. Per comparar-ho amb l'exercici 6 sense fiar-te de l'ull,
agafa els 20 píxels amb més importància
(`np.argsort(rf.feature_importances_)[-20:]`) i compta quants surten també a
`selector.get_support()`. El número que et doni és el que has d'explicar.

In [ ]:
# Exercici 7 (opcional)

# entrena un RandomForestClassifier(random_state=42) amb
# .fit(X_entrena, y_entrena)


# dibuixa rf.feature_importances_.reshape(8, 8) com una imatge (cmap="hot")